In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split

print("1. Loading V2.5 15-Minute Dataset...")
# 显式读取你仓库里已经做好的 V2.5 特征宽表
# 请确保路径正确，如果报错请调整相对于当前 Notebook 的路径
df_15min = pd.read_csv('../data/convertData/V2.5_15min_features.csv')

# 必须将时间戳转换为 datetime 格式，确保排序安全
df_15min['datetime'] = pd.to_datetime(df_15min['datetime'])
# 强制按时间线排序，这是时序任务的铁律
df_15min = df_15min.sort_values('datetime').reset_index(drop=True)

print("2. Defining 'High Volatility' Threshold...")
# 你的 V2.5 表里应该已经有 price_roll_std_24h 这一列了
# 如果没有，我们可以用真实的 price 计算一个短期的 6 小时滚动标准差来代表波动率
if 'price_roll_std_24h' not in df_15min.columns:
    print("   -> Creating Rolling Standard Deviation (Volatility)...")
    # 15分钟粒度，6小时 = 24 行
    df_15min['price_roll_std_6h'] = df_15min['price'].rolling(window=24).std()
else:
    df_15min['price_roll_std_6h'] = df_15min['price_roll_std_24h']

# 去除刚算滚动窗口时产生的头部 NaN 缺失值
df_15min = df_15min.dropna(subset=['price_roll_std_6h']).copy()

# 显式计算全市场波动的 85% 分位数（Top 15% 的疯狂时刻）作为警戒线
vol_threshold = df_15min['price_roll_std_6h'].quantile(0.85)
print(f"   -> Top 15% Volatility Threshold is strictly set at: {vol_threshold:.2f}")

print("3. Creating the Binary Classification Target (0 or 1)...")
# 生成我们要用来训练小分类器的标签：1 代表极端高波动，0 代表平稳
df_15min['is_high_volatility'] = (df_15min['price_roll_std_6h'] >= vol_threshold).astype(int)

# 打印标签分布情况，检查是否真的是 85% 对 15%
print("\nClass Distribution for High Volatility (1 = True, 0 = False):")
print(df_15min['is_high_volatility'].value_counts(normalize=True) * 100)

In [ ]:
print("4. Selecting Safe Features for the Classifier...")
# 我们只能给它看天气和时间周期，让它去猜会不会有高波动
# 注意：确保这些列名在你的 V2.5 表里是真实存在的！如果名字不对，请根据你的表头进行修改
vol_features = [
    'air_temp_mean', 
    'wind_speed_mean', 
    'temp_lag_24h', 
    'wind_lag_24h', 
    'hour', 
    'dayofweek'
]

# 分离出小模型的特征矩阵和考试答案
X_vol = df_15min[vol_features]
y_vol = df_15min['is_high_volatility']

# 划分出 80% 训练集用来教它，20% 不打乱用来验证
X_vol_train, X_vol_test, y_vol_train, y_vol_test = train_test_split(
    X_vol, y_vol, test_size=0.2, shuffle=False
)

print("5. Initializing and Training the XGBoost Risk Classifier...")
# 初始化 XGBoost 分类器 (注意这里是 XGBClassifier, 不是 Regressor)
# 我们不加 Optuna，直接用经验保守参数
risk_model = XGBClassifier(
    n_estimators=100,      # 建立100棵警戒树
    learning_rate=0.05,    # 学得慢一点，稳一点
    max_depth=5,           # 深度控制在5，防止死记硬背
    objective='binary:logistic', # 明确告诉它这是一个算概率的任务
    random_state=42        # 锁死随机种子，确保你每次运行结果完全一致
)

# 强行开机训练！
risk_model.fit(X_vol_train, y_vol_train)

print("6. Extracting the High-Volatility Probabilities...")
# 闭卷考试：但是我们不用 .predict() 让它输出 0 还是 1
# 我们用最精髓的 .predict_proba() 让它输出 0 到 1 之间柔和的百分比概率！
# predict_proba 返回两列：[变成0的概率, 变成1的概率]。我们需要取第二列 [:, 1]
predicted_probabilities = risk_model.predict_proba(X_vol)[:, 1]

# 7. 终极合体：把算出来的高价值概率，作为全新的一列写回我们的 V2.5 大表
df_15min['high_volatility_prob'] = predicted_probabilities

print("\n【SUCCESS!】 The ultimate 'high_volatility_prob' feature has been injected!")
# 显式查看最后的劳动成果：看它是不是真的算出了 0.XX 的概率
df_15min[['datetime', 'price', 'is_high_volatility', 'high_volatility_prob']].tail(10)

In [ ]:
print("8. Saving the new V3 Matrix (V2.5 + Risk Feature)...")
# 我们把包含了这个神级特征的表，专门另存为一个新文件
# 这样就不会污染你队友原本的 V2.5 表
save_path = '../data/convertData/V3.0_15min_Risk_Enhanced_Dataset.csv'
df_15min.to_csv(save_path, index=False)

print(f"Matrix officially saved to: {save_path}")
print("Ready for the Final Model Comparison!")